<!-- nav -->
[← TypeScript tutorial](README.md) · [Home](../../README.md) · [2 · Changing code safely →](02_Changing_Code_Safely.ipynb)

# 1 · Programs as data

**The problem.** Code generators usually paste strings together. It works until it doesn't: an expression pasted into
a template loses its meaning when it lands next to a stronger operator, a name collides with a keyword, an indent is
off by one. Here's the classic mistake, a generator that doubles an expression:

In [1]:
function doubledText(expression: string): string {
  return `def f(x):\n    return ${expression} * 2\n`;
}

console.log(doubledText("x + 1"));

def f(x):
    return x + 1 * 2



`x + 1 * 2` isn't twice `x + 1`. The generator needed to know Python's precedence, and so does every template that
pastes code. mbse-programs works on programs as **trees** instead: built, read and changed as data, and turned into
source text only at the end, by a printer that knows the language's grammar.

This case study parses Python source into a tree, looks at it, prints it, and builds one by hand. (Python is the
example; case study 5 does the same in C++ and TypeScript.)

In [2]:
import { Errors } from "@mbse/programs/Framework";
import { Python314, Syntax as P } from "@mbse/programs/Python";
import { outline } from "./toolkit.ts";

## Step 1: parsing source into a tree

A **standard**, here `Python314`, parses source text into a tree and prints trees back. `outline` (from the toolkit)
shows the tree one syntax node per line: its kind, and under it each child, labeled with the property that holds it.

In [3]:
const module = Python314.parse("def area(width, height):\n    return width * height\n") as P.Module;
console.log(outline(module));

Module
  body[0]: FunctionDef
    name: Identifier(spelling='area')
    args: Arguments
      args[0]: Arg
        arg: Identifier(spelling='width')
      args[1]: Arg
        arg: Identifier(spelling='height')
    body[0]: Return
      value: BinOp(op='*')
        left: Name
          id: Identifier(spelling='width')
        right: Name
          id: Identifier(spelling='height')


The kinds are Python's own abstract syntax, as Python's `ast` module has it: `FunctionDef`, `Arguments`, `Arg`,
`Return`, `BinOp`, `Name`. Where it departs from `ast`, it keeps what `ast` drops: names are `Identifier` syntax nodes,
so one traversal finds every name, and literals keep their spelling.

## Step 2: syntax nodes are plain objects

Each syntax node is an ordinary object with its properties. **Attributes** hold values (`op='*'`, an identifier's
`spelling`), and **children** hold other syntax nodes:

In [4]:
const fn = module.body[0] as P.FunctionDef;
console.log(fn.name.spelling, fn.args.args.map((arg) => arg.arg.spelling));
const product = (fn.body[0] as P.Return).value as P.BinOp;
console.log(product.kind().KIND, product.op, (product.left as P.Name).id.spelling, (product.right as P.Name).id.spelling);

area [ "width", "height" ]


BinOp * width height


## Step 3: printing

Printing writes one fixed layout: four spaces per level, PEP 8's blank lines, parentheses only where the grammar
needs them. The tree doesn't record whitespace, so source comes back tidied:

In [5]:
const messy = Python314.parse("def  area( width,height ) :\n  return (width*height)\nx=area(2,3);y=area(4,5)\n");
console.log(Python314.print(messy));

def area(width, height):
    return (width * height)


x = area(2, 3)
y = area(4, 5)



The parentheses around `width*height` were written in the source, so the tree keeps them (as a `Parenthesized`
node); everything else is the printer's layout.

## Step 4: building a tree

Trees are built with each language's fluent **builders**, `P.LANGUAGE.Builders.<Kind>()`, with one setter per property,
typed from the language. A child's setter takes a syntax node, or a function that builds one. Here's the doubling
generator again, written as a tree: it takes the expression as a tree too, and multiplies it by two.

In [6]:
const Py = P.LANGUAGE.Builders;

function doubled(expression: P.Expression): P.FunctionDef {
  const body = Py.Return().value((b) => b.BinOp().left(expression).op("*").right((b) => b.Constant().spelling("2")));
  return Py.FunctionDef().name("f").args((a) => a.add_args((b) => b.arg("x"))).add_body(body).create();
}

const x_plus_1 = ((Python314.parse("x + 1\n") as P.Module).body[0] as P.Expr).value;
console.log(Python314.print(doubled(x_plus_1)));

def f(x):
    return (x + 1) * 2


The printer added the parentheses, because a `BinOp` of `+` under a `*` needs them. The generator didn't have to know.
A property whose kind has one attribute takes the attribute's value directly: `.name("f")` makes the `Identifier`, and
`.arg("x")` the parameter's.

## Step 5: a tree must be complete

`validate` reports what's missing or wrong, by path. A tree that isn't valid doesn't print:

In [7]:
const halfBuilt = Py.BinOp().op("+").right((b) => b.Name().id("y")).create();
console.log(P.LANGUAGE.validate(halfBuilt));
try {
  Python314.print(halfBuilt);
} catch (error) {
  console.log((error as Error).message);
}

[ "a BinOp needs a left" ]


a BinOp needs a left


## Why it works this way

- **Text only at the edges.** Parsing and printing are the only places text appears. Everything in between, reading,
  changing and generating code, works on trees, so precedence, quoting and layout are the printer's job, done once.
- **The language's own abstract syntax.** Python's tree follows `ast`; TypeScript's follows typescript-estree, and C++'s
  the standard's grammar (case study 5). A programmer who knows the language's tools knows its tree.
- **Plain objects.** Syntax nodes are ordinary mutable objects, so a program reads and edits them like any other
  data: the next case study changes code in place.

<!-- nav -->
[← TypeScript tutorial](README.md) · [Home](../../README.md) · [2 · Changing code safely →](02_Changing_Code_Safely.ipynb)